This notebook demonstrates how to run the forest deforestation User Defined Process

In [1]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

INFO:openeo.config:Loaded openEO client config from sources: []


In [3]:
connection.authenticate_oidc()

INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [4]:
job = connection.job("j-2611231717074e30ad0dab8293ea4f6a") # change job, take from s1 processing
print(job.status())
meta = job.get_results().get_metadata()
print(meta.get("type"), meta.keys())

features = meta.get("features") or [meta]
for item in features:
    print(item.get("bbox"))
    props = item.get("properties", {})
    print(props.get("datetime"), props.get("start_datetime"), props.get("end_datetime"))
    print(list(item.get("assets", {})))

finished
Collection dict_keys(['assets', 'description', 'extent', 'id', 'license', 'links', 'openeo:status', 'providers', 'stac_extensions', 'stac_version', 'summaries', 'type'])
None
None None None
['openEO.tif']


In [5]:
import json
print(json.dumps(meta["extent"], indent=2))
for link in meta["links"]:
    print(link.get("rel"), link.get("title"), link.get("href", "")[:160])
print(json.dumps(meta["assets"]["openEO.tif"], indent=2)[:2000])

{
  "spatial": {
    "bbox": [
      [
        32.637423,
        0.060209,
        33.131945,
        0.291267
      ]
    ]
  },
  "temporal": {
    "interval": [
      [
        "2019-10-01T00:00:00Z",
        "2025-04-01T00:00:00Z"
      ]
    ]
  }
}
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2023/04/23/S1A_IW_GRDH_1SDV_20230423T160519_20230423T160544_048225_05CC89_0DAD_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2023/04/23/S1A_IW_GRDH_1SDV_20230423T160519_20230423T160544_048225_05CC89_0DAD_COG.SAFE
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2021/05/27/S1A_IW_GRDH_1SDV_20210527T160509_20210527T160534_038075_047E63_F8B7_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2021/05/27/S1A_IW_GRDH_1SDV_20210527T160509_20210527T160534_038075_047E63_F8B7_COG.SAFE
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2023/10/20/S1A_IW_GRDH_1SDV_20231020T160527_20231020T160552_050850_062118_6727_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-CO

In [6]:
spatial_extent = {
    "west": 32.637423,
    "south": 0.060209,
    "east": 33.131945,
    "north": 0.291267,
}

resample_spatial_resolution = 30  # m

temporal_variability_threshold = 0.6  # units: dB
flattening_threshold = 0.18  # units: dimensionless
logistic_sse_threshold = 80  # units: dB^2

min_connected_area = 10000  # m^2

In [7]:
# Do not override the thresholds above. This used to set flattening back to 0.14
# after the parameter cell, so the job ignored the values entered there.


## Sentinel 1

In [ ]:
# load results from previous batch job
JOB_ID = "j-2611231717074e30ad0dab8293ea4f6a"  # change job, take from s1 processing

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    temporal_extent=["2019-10-01", "2025-04-01"],
    bands=["sd", "p05", "p95", "min_sse", "min_sse_t"],
)

In [9]:
# fix for nodata tiling artifacts
# https://forum.dataspace.copernicus.eu/t/nodata-tiling-artifacts/5732/6

s1_dB.result_node().update_arguments(featureflags={"retain_nodata_tiles": True})

In [10]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

# load forest baseline

In [11]:
# load results from previous batch job
JOB_ID = "j-26112317162446ec99e059819fcf7265"  # change job, take from forest baseline processing

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    # note: important to specify temporal_extent here
    # otherwise openEO writes STAC temporal interval [null, null]
    temporal_extent="2020-01-01",
    bands=["B0"],
)

In [12]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

# Run deforestation UDP

In [13]:
cube = connection.datacube_from_process(
    "deforestation",
    namespace=urls.DEFORESTATION_UDP,
    spatial_extent=spatial_extent,
    forest_baseline_datacube=forest_baseline_mask,
    sentinel_1_datacube=s1_dB,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
    logistic_sse_threshold=logistic_sse_threshold,
    min_connected_area=min_connected_area,
)

In [14]:
job = cube.create_job(out_format="GTiff")
# job = cube.create_job(out_format="netCDF")

In [15]:
job.start_and_wait()

0:00:00 Job 'j-26112410011843848d3156279eb44900': send 'start'
0:00:05 Job 'j-26112410011843848d3156279eb44900': created (progress 0%)
0:00:10 Job 'j-26112410011843848d3156279eb44900': queued (progress 0%)
0:00:16 Job 'j-26112410011843848d3156279eb44900': queued (progress 0%)
0:00:24 Job 'j-26112410011843848d3156279eb44900': queued (progress 0%)
0:00:34 Job 'j-26112410011843848d3156279eb44900': queued (progress 0%)
0:00:47 Job 'j-26112410011843848d3156279eb44900': queued (progress 0%)
0:01:02 Job 'j-26112410011843848d3156279eb44900': running (progress N/A)
0:01:21 Job 'j-26112410011843848d3156279eb44900': running (progress N/A)
0:01:45 Job 'j-26112410011843848d3156279eb44900': error (progress N/A)
Your batch job 'j-26112410011843848d3156279eb44900' failed. Error logs:
[{'id': '[1795514570763, 240539]', 'time': '2026-11-24T10:02:50.763Z', 'level': 'error', 'message': 'OpenEO batch job failed: NoDataAvailableException(status_code=400, code=\'NoDataAvailable\', message="No data available 

JobFailedException: Batch job 'j-26112410011843848d3156279eb44900' didn't finish successfully. Status: error (after 0:01:46).

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-udp/
!rm -r output-udp/

In [ ]:
results.download_files("output-udp/")

INFO:openeo.rest.job:Downloading job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26112315221346968fd689d40e190f90/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=43107f6412574ef48bf08a37325eff59%2F20261123%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20261123T152653Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYxMTIzMTUyMjEzNDY5NjhmZDY4OWQ0MGUxOTBmOTAiXSwidXNlcl9pZCI6WyIxM2YyYTg1Mi00YzFjLTQ4N2EtOGExYS0xYzA1YjE2M2I4YTIiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNlLmNvcGVybmljdXMu

[PosixPath('output-udp/openEO.tif'), PosixPath('output-udp/job-results.json')]

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)